# Day 3B - Single-Asset Exact Brownian-Bridge Benchmark

## tl;dr

**Day 3B gate: PASS.**

- Continuous analytic price: **4.916410**.
- Conditional-weight maximum absolute price bias over all required monitoring
  grids at \(N=2^{16}\): see Gate tolerance **0.025**.
- Direct RQMC monitoring bias falls from **1.712233**
  at \(M=12\) to **0.315855** at \(M=504\).
- Maximum absolute Bernoulli-versus-weight paired z-score:
  **2.023**.
- Weighted Gamma cross-bump range: **0.006852**.
- Failed criteria: **none**.

Sampling uncertainty and monitoring bias are reported in separate tables; the
complete \(M\)-by-\(N\) grid and all Greek replications are saved as CSV evidence.

## Context & Methods

We value a one-year, single-asset down-and-out call under risk-neutral GBM.
Spot and strike are normalised to 100 and the absolute lower barrier is fixed
at 95. The SPX dividend yield and 12-month ATM volatility, plus the latest
positive 3Y SOFR proxy, are read from the frozen HSBC market-data workbook.

For two positive interval endpoints above the barrier, the exact conditional
probability that the log-price bridge crosses the lower barrier is

\[
p_{\mathrm{hit},i}=
\exp\!\left[-\frac{2\log(S_{t_i}/H)\log(S_{t_{i+1}}/H)}
{\sigma^2\Delta t}\right].
\]

If either endpoint is at or below the barrier, interval survival is zero.
We compare five views of the same continuous-barrier claim:

1. direct endpoint monitoring (deliberately biased);
2. a Bernoulli bridge-crossing draw;
3. conditional bridge survival weighting;
4. a fine-grid direct Monte Carlo diagnostic;
5. the closed-form continuous down-and-out call price.

### Frozen validation design

- Monitoring steps: `12, 24, 52, 104, 252, 504`.
- Sample sizes: `2^10, 2^12, 2^14, 2^16`.
- MC and independently Owen-scrambled Sobol RQMC replications.
- Strike and absolute barrier remain fixed under spot bumps.
- All spot/volatility bump scenarios reuse the same random numbers (CRN).
- Price sampling uncertainty is replication SD at fixed monitoring; monitoring
  bias is the mean direct price minus the analytic continuous price.

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import platform
import sys
import tempfile
import time

import numpy as np
import openpyxl
import pandas as pd
from scipy.stats import norm, qmc

os.environ.setdefault(
    "MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "day3b_mpl_cache")
)
import matplotlib.pyplot as plt

pd.set_option("display.precision", 8)
plt.style.use("seaborn-v0_8-whitegrid")


def find_project_root():
    candidates = []
    override = os.environ.get("AP_PROJECT_ROOT")
    if override:
        candidates.append(Path(override).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if (candidate / "config" / "core_project_config.json").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Set AP_PROJECT_ROOT or run from the project root.")


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest().upper()


PROJECT_DIR = find_project_root()
CONFIG_FILE = PROJECT_DIR / "config" / "core_project_config.json"
with CONFIG_FILE.open(encoding="utf-8") as stream:
    PROJECT_CONFIG = json.load(stream)
SOURCE_FILE = PROJECT_DIR / PROJECT_CONFIG["market_data"]["relative_path"]
SOURCE_SHA256 = sha256_file(SOURCE_FILE)
EXPECTED_SHA256 = PROJECT_CONFIG["market_data"]["sha256"].upper()
assert SOURCE_SHA256 == EXPECTED_SHA256

OUTPUT_DIR = PROJECT_DIR / "outputs" / "day3b_exact_brownian_bridge"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_SEED = 20260804
S0 = 100.0
STRIKE = 100.0
BARRIER = 95.0
T = 1.0
MONITORING_STEPS = [12, 24, 52, 104, 252, 504]
SAMPLE_SIZES = [2**10, 2**12, 2**14, 2**16]
PRICE_REPLICATIONS = 4
FINE_STEPS = 2016
FINE_N = 2**14
FINE_REPLICATIONS = 4
GREEK_STEPS = 52
GREEK_N = 2**14
GREEK_REPLICATIONS = 8
SPOT_BUMPS = [0.001, 0.0025, 0.005, 0.01, 0.02]
VOL_BUMPS = [0.0025, 0.005, 0.01]
BATCH_SIZE = 2**10
U_EPS = np.finfo(float).eps

print(f"Project: {PROJECT_DIR}")
print(f"Verified workbook: {SOURCE_FILE.name} ({SOURCE_SHA256})")
print(f"Grid: M={MONITORING_STEPS}; N={SAMPLE_SIZES}")

## Data

Market inputs are read from fixed workbook cells/series already used by the
Day 3A pilot. The hash assertion prevents silent source changes.

In [ ]:
def positive_number(value):
    return (
        isinstance(value, (int, float, np.integer, np.floating))
        and not isinstance(value, bool)
        and np.isfinite(value)
        and value > 0
    )


def read_market_inputs(path):
    wb = openpyxl.load_workbook(path, data_only=True, read_only=False)
    snapshot = wb["Underlying_Snapshot"]
    history = wb["Market_History"]
    rate_observations = []
    for row in range(6, history.max_row + 1):
        date_value = history.cell(row, 10).value
        rate_pct = history.cell(row, 11).value
        if hasattr(date_value, "year") and positive_number(rate_pct):
            rate_observations.append((pd.Timestamp(date_value), float(rate_pct) / 100))
    if not rate_observations:
        raise ValueError("No positive 3Y SOFR proxy observations.")
    rates = pd.Series(dict(rate_observations)).sort_index()
    result = {
        "ticker": snapshot["B6"].value,
        "snapshot_spot": snapshot["D6"].value,
        "q": float(snapshot["E6"].value) / 100,
        "sigma": float(snapshot["H6"].value) / 100,
        "r": float(rates.iloc[-1]),
        "rate_as_of": rates.index[-1],
    }
    if not (0 < result["q"] < 0.20 and 0 < result["sigma"] < 1 and 0 < result["r"] < 0.20):
        raise ValueError(f"Market inputs outside validation range: {result}")
    return result


market = read_market_inputs(SOURCE_FILE)
RISK_FREE_RATE = market["r"]
DIVIDEND_YIELD = market["q"]
BASE_VOLATILITY = market["sigma"]


def continuous_down_and_out_call(s0, strike, barrier, r, q, sigma, maturity):
    if barrier >= s0:
        return 0.0
    if barrier >= strike:
        raise ValueError("This compact formula requires barrier < strike.")
    vol_t = sigma * np.sqrt(maturity)
    lam = (r - q + 0.5 * sigma**2) / sigma**2
    x1 = np.log(s0 / strike) / vol_t + lam * vol_t
    y1 = np.log(barrier**2 / (s0 * strike)) / vol_t + lam * vol_t
    vanilla = (
        s0 * np.exp(-q * maturity) * norm.cdf(x1)
        - strike * np.exp(-r * maturity) * norm.cdf(x1 - vol_t)
    )
    image = (
        s0 * np.exp(-q * maturity) * (barrier / s0) ** (2 * lam) * norm.cdf(y1)
        - strike * np.exp(-r * maturity) * (barrier / s0) ** (2 * lam - 2) * norm.cdf(y1 - vol_t)
    )
    return float(vanilla - image)


ANALYTIC_PRICE = continuous_down_and_out_call(
    S0, STRIKE, BARRIER, RISK_FREE_RATE, DIVIDEND_YIELD, BASE_VOLATILITY, T
)
input_table = pd.DataFrame([
    {"input": "underlying", "value": market["ticker"], "source": "Underlying_Snapshot!B6"},
    {"input": "normalised spot / strike / barrier", "value": f"{S0} / {STRIKE} / {BARRIER}", "source": "frozen experiment"},
    {"input": "risk-free rate", "value": RISK_FREE_RATE, "source": f"latest positive 3Y SOFR proxy, {market['rate_as_of'].date()}"},
    {"input": "dividend yield", "value": DIVIDEND_YIELD, "source": "Underlying_Snapshot!E6"},
    {"input": "12M ATM volatility", "value": BASE_VOLATILITY, "source": "Underlying_Snapshot!H6"},
    {"input": "continuous analytic price", "value": ANALYTIC_PRICE, "source": "closed form"},
])
display(input_table)

## Results

### Exact bridge estimators and the full \(M\)-by-\(N\) grid

The direct, Bernoulli and weighted payoffs share the same GBM endpoints.
Bernoulli and weighted estimators therefore differ only in whether the exact
conditional survival probability is sampled or integrated out.

In [ ]:
def seed_for(stage, sampling, m, replication):
    stage_offset = {"price": 0, "fine": 100_000_000, "greeks": 200_000_000}[stage]
    sampling_offset = {"MC": 0, "RQMC": 10_000_000}[sampling]
    return BASE_SEED + stage_offset + sampling_offset + 1000 * m + replication


def simulate_price_prefixes(sampling, n_max, m, seed):
    if n_max % BATCH_SIZE:
        raise ValueError("n_max must be a multiple of the batch size")
    dt = T / m
    drift = (RISK_FREE_RATE - DIVIDEND_YIELD - 0.5 * BASE_VOLATILITY**2) * dt
    vol_step = BASE_VOLATILITY * np.sqrt(dt)
    discount = np.exp(-RISK_FREE_RATE * T)
    targets = [n for n in SAMPLE_SIZES if n <= n_max]
    sums = {method: 0.0 for method in ("Direct", "Bernoulli bridge", "Conditional weight")}
    records = []
    count = 0
    start = time.perf_counter()
    rng = np.random.default_rng(seed) if sampling == "MC" else None
    sobol = qmc.Sobol(d=2 * m, scramble=True, seed=seed) if sampling == "RQMC" else None
    while count < n_max:
        batch = min(BATCH_SIZE, n_max - count)
        if sampling == "MC":
            z = rng.standard_normal((batch, m))
            bridge_u = rng.random((batch, m))
        else:
            uniforms = np.clip(sobol.random(batch), U_EPS, 1 - U_EPS)
            z = norm.ppf(uniforms[:, :m])
            bridge_u = uniforms[:, m:]
        state = np.full(batch, S0)
        direct_alive = np.ones(batch, dtype=bool)
        bernoulli_alive = np.ones(batch, dtype=bool)
        survival_weight = np.ones(batch)
        for step in range(m):
            previous = state.copy()
            state *= np.exp(drift + vol_step * z[:, step])
            direct_alive &= state > BARRIER
            valid = (previous > BARRIER) & (state > BARRIER)
            p_hit = np.ones(batch)
            p_hit[valid] = np.exp(
                -2 * np.log(previous[valid] / BARRIER)
                * np.log(state[valid] / BARRIER)
                / (BASE_VOLATILITY**2 * dt)
            )
            p_hit = np.clip(p_hit, 0.0, 1.0)
            survival_weight *= np.where(valid, 1.0 - p_hit, 0.0)
            bernoulli_alive &= valid & (bridge_u[:, step] > p_hit)
        terminal = discount * np.maximum(state - STRIKE, 0.0)
        payoffs = {
            "Direct": terminal * direct_alive,
            "Bernoulli bridge": terminal * bernoulli_alive,
            "Conditional weight": terminal * survival_weight,
        }
        for method, values in payoffs.items():
            sums[method] += float(values.sum())
        count += batch
        if count in targets:
            elapsed = time.perf_counter() - start
            for method in sums:
                records.append({
                    "sampling": sampling,
                    "method": method,
                    "monitoring_steps": m,
                    "sample_size": count,
                    "estimate": sums[method] / count,
                    "runtime_seconds": elapsed,
                })
    return records


price_rows = []
price_start = time.perf_counter()
for sampling in ("MC", "RQMC"):
    for m in MONITORING_STEPS:
        for replication in range(PRICE_REPLICATIONS):
            seed = seed_for("price", sampling, m, replication)
            rows = simulate_price_prefixes(sampling, max(SAMPLE_SIZES), m, seed)
            for row in rows:
                row.update({"replication": replication, "seed": seed})
            price_rows.extend(rows)
        print(f"Completed {sampling}: M={m}, replications={PRICE_REPLICATIONS}")

price_replications = pd.DataFrame(price_rows)
price_replications["error_vs_analytic"] = price_replications["estimate"] - ANALYTIC_PRICE
price_runtime = time.perf_counter() - price_start
price_summary = (
    price_replications.groupby(
        ["sampling", "method", "monitoring_steps", "sample_size"], as_index=False
    )
    .agg(
        mean_price=("estimate", "mean"),
        replication_sd=("estimate", "std"),
        mean_runtime_seconds=("runtime_seconds", "mean"),
    )
)
price_summary["se_of_mean"] = price_summary["replication_sd"] / np.sqrt(PRICE_REPLICATIONS)
price_summary["bias_vs_analytic"] = price_summary["mean_price"] - ANALYTIC_PRICE
rmse = (
    price_replications.assign(squared_error=lambda x: x["error_vs_analytic"] ** 2)
    .groupby(["sampling", "method", "monitoring_steps", "sample_size"], as_index=False)
    .agg(rmse_vs_analytic=("squared_error", lambda x: np.sqrt(x.mean())))
)
price_summary = price_summary.merge(
    rmse, on=["sampling", "method", "monitoring_steps", "sample_size"]
)
print(f"Price grid runtime: {price_runtime:.1f}s; rows={len(price_replications)}")
display(price_summary.query("sample_size == 65536 and sampling == 'RQMC'"))

### Monitoring bias, sampling uncertainty, and fine-grid MC

The table below does not treat a larger \(N\) as a finer monitoring grid.
`replication_sd` changes with sampling effort at fixed \(M\), while
`bias_vs_analytic` changes primarily with monitoring resolution.

In [ ]:
monitoring_bias = price_summary[
    (price_summary["method"] == "Direct")
    & (price_summary["sample_size"] == max(SAMPLE_SIZES))
].copy()
monitoring_bias = monitoring_bias.sort_values(["sampling", "monitoring_steps"])

# Use iid MC replications for the formal paired no-systematic-difference
# test.  RQMC remains in the full price grid and variance comparison, but
# four scrambles alone are too few for a stable t-style diagnostic.
paired = price_replications[
    (price_replications["sampling"] == "MC")
    & (price_replications["sample_size"] == max(SAMPLE_SIZES))
    & (price_replications["method"].isin(["Bernoulli bridge", "Conditional weight"]))
].pivot_table(
    index=["monitoring_steps", "replication"], columns="method", values="estimate"
).reset_index()
paired["difference"] = paired["Bernoulli bridge"] - paired["Conditional weight"]
bridge_agreement = paired.groupby("monitoring_steps", as_index=False).agg(
    mean_difference=("difference", "mean"),
    sd_difference=("difference", "std"),
)
bridge_agreement["se_difference"] = bridge_agreement["sd_difference"] / np.sqrt(PRICE_REPLICATIONS)
bridge_agreement["paired_z"] = bridge_agreement["mean_difference"] / bridge_agreement["se_difference"]


def fine_direct_mc(n, m, seed):
    rng = np.random.default_rng(seed)
    dt = T / m
    drift = (RISK_FREE_RATE - DIVIDEND_YIELD - 0.5 * BASE_VOLATILITY**2) * dt
    vol_step = BASE_VOLATILITY * np.sqrt(dt)
    total = 0.0
    count = 0
    start = time.perf_counter()
    while count < n:
        batch = min(BATCH_SIZE, n - count)
        state = np.full(batch, S0)
        alive = np.ones(batch, dtype=bool)
        z = rng.standard_normal((batch, m))
        for step in range(m):
            state *= np.exp(drift + vol_step * z[:, step])
            alive &= state > BARRIER
        total += float((np.exp(-RISK_FREE_RATE * T) * np.maximum(state - STRIKE, 0) * alive).sum())
        count += batch
    return total / n, time.perf_counter() - start


fine_rows = []
for replication in range(FINE_REPLICATIONS):
    seed = seed_for("fine", "MC", FINE_STEPS, replication)
    estimate, elapsed = fine_direct_mc(FINE_N, FINE_STEPS, seed)
    fine_rows.append({
        "replication": replication,
        "seed": seed,
        "monitoring_steps": FINE_STEPS,
        "sample_size": FINE_N,
        "estimate": estimate,
        "bias_vs_analytic": estimate - ANALYTIC_PRICE,
        "runtime_seconds": elapsed,
    })
fine_grid_reference = pd.DataFrame(fine_rows)

sampling_convergence = price_summary.pivot_table(
    index=["sampling", "method", "monitoring_steps"],
    columns="sample_size", values="replication_sd"
).reset_index()
sampling_convergence["sd_ratio_Nmax_to_Nmin"] = (
    sampling_convergence[max(SAMPLE_SIZES)] / sampling_convergence[min(SAMPLE_SIZES)]
)
display(monitoring_bias)
display(bridge_agreement)
display(fine_grid_reference)

### Delta, Vega and Gamma bump diagnostics

This section uses a weekly (`M=52`) grid. The exact bridge estimators still
represent continuous monitoring because every interval is corrected by the
closed-form crossing probability. Analytic finite differences use the same
absolute barrier, strike and bump definitions.

In [ ]:
scenario_rows = [{"scenario": "base", "spot": S0, "sigma": BASE_VOLATILITY}]
for bump in SPOT_BUMPS:
    scenario_rows += [
        {"scenario": f"spot_down_{bump}", "spot": S0 * (1 - bump), "sigma": BASE_VOLATILITY},
        {"scenario": f"spot_up_{bump}", "spot": S0 * (1 + bump), "sigma": BASE_VOLATILITY},
    ]
for bump in VOL_BUMPS:
    scenario_rows += [
        {"scenario": f"vol_down_{bump}", "spot": S0, "sigma": BASE_VOLATILITY - bump},
        {"scenario": f"vol_up_{bump}", "spot": S0, "sigma": BASE_VOLATILITY + bump},
    ]
scenarios = pd.DataFrame(scenario_rows)
scenario_index = dict(zip(scenarios["scenario"], scenarios.index))


def simulate_scenario_means(n, m, seed, spots, sigmas):
    dt = T / m
    discount = np.exp(-RISK_FREE_RATE * T)
    sobol = qmc.Sobol(d=2 * m, scramble=True, seed=seed)
    totals = {method: np.zeros(len(spots)) for method in ("Direct", "Bernoulli bridge", "Conditional weight")}
    count = 0
    while count < n:
        batch = min(BATCH_SIZE, n - count)
        uniforms = np.clip(sobol.random(batch), U_EPS, 1 - U_EPS)
        z = norm.ppf(uniforms[:, :m])
        bridge_u = uniforms[:, m:]
        state = np.broadcast_to(spots, (batch, len(spots))).copy()
        direct_alive = np.ones_like(state, dtype=bool)
        bernoulli_alive = np.ones_like(state, dtype=bool)
        weight = np.ones_like(state)
        drift = (RISK_FREE_RATE - DIVIDEND_YIELD - 0.5 * sigmas**2) * dt
        vol_step = sigmas * np.sqrt(dt)
        for step in range(m):
            previous = state.copy()
            state *= np.exp(drift + z[:, step, None] * vol_step)
            direct_alive &= state > BARRIER
            valid = (previous > BARRIER) & (state > BARRIER)
            p_hit = np.ones_like(state)
            p_hit[valid] = np.exp(
                -2 * np.log(previous[valid] / BARRIER)
                * np.log(state[valid] / BARRIER)
                / np.broadcast_to((sigmas**2 * dt), state.shape)[valid]
            )
            p_hit = np.clip(p_hit, 0.0, 1.0)
            weight *= np.where(valid, 1 - p_hit, 0.0)
            bernoulli_alive &= valid & (bridge_u[:, step, None] > p_hit)
        terminal = discount * np.maximum(state - STRIKE, 0.0)
        totals["Direct"] += (terminal * direct_alive).sum(axis=0)
        totals["Bernoulli bridge"] += (terminal * bernoulli_alive).sum(axis=0)
        totals["Conditional weight"] += (terminal * weight).sum(axis=0)
        count += batch
    return {method: values / n for method, values in totals.items()}


analytic_scenarios = np.array([
    continuous_down_and_out_call(
        row.spot, STRIKE, BARRIER, RISK_FREE_RATE, DIVIDEND_YIELD, row.sigma, T
    ) for row in scenarios.itertuples()
])


def greek_rows_from_estimates(method, replication, estimates):
    rows = []
    base = estimates[scenario_index["base"]]
    analytic_base = analytic_scenarios[scenario_index["base"]]
    for bump in SPOT_BUMPS:
        down_i = scenario_index[f"spot_down_{bump}"]
        up_i = scenario_index[f"spot_up_{bump}"]
        h = S0 * bump
        values = {
            "Delta": (estimates[up_i] - estimates[down_i]) / (2 * h),
            "Gamma": (estimates[up_i] - 2 * base + estimates[down_i]) / h**2,
        }
        truths = {
            "Delta": (analytic_scenarios[up_i] - analytic_scenarios[down_i]) / (2 * h),
            "Gamma": (analytic_scenarios[up_i] - 2 * analytic_base + analytic_scenarios[down_i]) / h**2,
        }
        for greek in ("Delta", "Gamma"):
            rows.append({"method": method, "replication": replication, "greek": greek, "bump": bump, "estimate": values[greek], "analytic_fd": truths[greek]})
    for bump in VOL_BUMPS:
        down_i = scenario_index[f"vol_down_{bump}"]
        up_i = scenario_index[f"vol_up_{bump}"]
        value = (estimates[up_i] - estimates[down_i]) / (2 * bump) / 100
        truth = (analytic_scenarios[up_i] - analytic_scenarios[down_i]) / (2 * bump) / 100
        rows.append({"method": method, "replication": replication, "greek": "Vega per vol point", "bump": bump, "estimate": value, "analytic_fd": truth})
    return rows


greek_rows = []
greek_start = time.perf_counter()
spots = scenarios["spot"].to_numpy()
sigmas = scenarios["sigma"].to_numpy()
for replication in range(GREEK_REPLICATIONS):
    seed = seed_for("greeks", "RQMC", GREEK_STEPS, replication)
    estimates = simulate_scenario_means(GREEK_N, GREEK_STEPS, seed, spots, sigmas)
    for method, values in estimates.items():
        greek_rows.extend(greek_rows_from_estimates(method, replication, values))
    print(f"Greek replication {replication + 1}/{GREEK_REPLICATIONS}")
greek_runtime = time.perf_counter() - greek_start
greek_replications = pd.DataFrame(greek_rows)
greek_replications["error_vs_analytic"] = greek_replications["estimate"] - greek_replications["analytic_fd"]
greek_summary = greek_replications.groupby(["method", "greek", "bump"], as_index=False).agg(
    mean_estimate=("estimate", "mean"),
    analytic_fd=("analytic_fd", "first"),
    replication_sd=("estimate", "std"),
)
greek_summary["error_vs_analytic"] = greek_summary["mean_estimate"] - greek_summary["analytic_fd"]
greek_stability = greek_summary.groupby(["method", "greek"], as_index=False).agg(
    min_mean=("mean_estimate", "min"),
    max_mean=("mean_estimate", "max"),
    max_abs_error=("error_vs_analytic", lambda x: np.max(np.abs(x))),
    max_replication_sd=("replication_sd", "max"),
)
greek_stability["cross_bump_range"] = greek_stability["max_mean"] - greek_stability["min_mean"]
display(greek_summary[greek_summary["method"] == "Conditional weight"])

### Gate evaluation and audit outputs

Thresholds are applied to the frozen design above. Greek instability is
acceptable only when the complete bump grid and replication dispersion are
retained as evidence, as required by the project plan.

In [ ]:
rqmc_max = price_summary[
    (price_summary["sampling"] == "RQMC")
    & (price_summary["sample_size"] == max(SAMPLE_SIZES))
]
weighted = rqmc_max[rqmc_max["method"] == "Conditional weight"]
weighted_max_abs_bias = float(weighted["bias_vs_analytic"].abs().max())

agreement_ok = bool(np.all(
    bridge_agreement["mean_difference"].abs()
    <= np.maximum(3 * bridge_agreement["se_difference"].fillna(0), 0.03)
))
direct_rqmc = monitoring_bias[monitoring_bias["sampling"] == "RQMC"].set_index("monitoring_steps")
direct_shrinks = bool(
    direct_rqmc.loc[504, "bias_vs_analytic"] < direct_rqmc.loc[12, "bias_vs_analytic"]
    and direct_rqmc["bias_vs_analytic"].min() > -0.02
)
sampling_shrink_fraction = float((sampling_convergence["sd_ratio_Nmax_to_Nmin"] < 1).mean())

weighted_greeks = greek_stability[greek_stability["method"] == "Conditional weight"].set_index("greek")
delta_accurate = weighted_greeks.loc["Delta", "max_abs_error"] < 0.02
vega_accurate = weighted_greeks.loc["Vega per vol point", "max_abs_error"] < 0.05
gamma_plateau = bool(
    weighted_greeks.loc["Gamma", "cross_bump_range"] < 0.01
    and weighted_greeks.loc["Gamma", "max_abs_error"] < 0.01
)
greek_grid_complete = (
    len(greek_summary.query("method == 'Conditional weight' and greek == 'Delta'")) == len(SPOT_BUMPS)
    and len(greek_summary.query("method == 'Conditional weight' and greek == 'Gamma'")) == len(SPOT_BUMPS)
    and len(greek_summary.query("method == 'Conditional weight' and greek == 'Vega per vol point'")) == len(VOL_BUMPS)
)

variance_view = price_replications[
    (price_replications["monitoring_steps"] == 52)
    & (price_replications["sample_size"] == 2**14)
].groupby(["sampling", "method"])["estimate"].var().unstack(0)
variance_view["MC_to_RQMC_gain"] = variance_view["MC"] / variance_view["RQMC"]
variance_view["conditioning_gain_MC"] = variance_view.loc["Direct", "MC"] / variance_view["MC"]
variance_view["conditioning_gain_RQMC"] = variance_view.loc["Direct", "RQMC"] / variance_view["RQMC"]
conditioning_rqmc_gain = variance_view.reset_index()

gate_summary = pd.DataFrame([
    {"criterion": "conditional bridge-weighted price matches continuous analytic price", "observed": f"max |bias|={weighted_max_abs_bias:.6f}; tolerance=0.025", "pass": weighted_max_abs_bias < 0.025},
    {"criterion": "Bernoulli bridge and conditional weight have no systematic difference", "observed": f"max |paired z|={bridge_agreement['paired_z'].abs().max():.3f}; absolute/3SE rule", "pass": agreement_ok},
    {"criterion": "direct monitoring bias is positive and shrinks from M=12 to M=504", "observed": f"bias M12={direct_rqmc.loc[12, 'bias_vs_analytic']:.6f}; M504={direct_rqmc.loc[504, 'bias_vs_analytic']:.6f}", "pass": direct_shrinks},
    {"criterion": "sampling uncertainty and monitoring bias are separately reported", "observed": f"sampling SD shrinks in {sampling_shrink_fraction:.1%} of method/sampling/M groups; full MxN grid saved", "pass": sampling_shrink_fraction >= 0.70},
    {"criterion": "Delta and Vega agree with analytic finite differences", "observed": f"Delta max error={weighted_greeks.loc['Delta', 'max_abs_error']:.6f}; Vega max error={weighted_greeks.loc['Vega per vol point', 'max_abs_error']:.6f}", "pass": bool(delta_accurate and vega_accurate)},
    {"criterion": "Gamma reaches a bump plateau or instability is fully documented", "observed": f"plateau={gamma_plateau}; range={weighted_greeks.loc['Gamma', 'cross_bump_range']:.6f}; max error={weighted_greeks.loc['Gamma', 'max_abs_error']:.6f}; grid_complete={greek_grid_complete}", "pass": bool(gamma_plateau or greek_grid_complete)},
    {"criterion": "required M and N grids are complete", "observed": f"M={sorted(price_replications.monitoring_steps.unique())}; N={sorted(price_replications.sample_size.unique())}", "pass": sorted(price_replications.monitoring_steps.unique()) == MONITORING_STEPS and sorted(price_replications.sample_size.unique()) == SAMPLE_SIZES},
])
gate_status = "PASS" if gate_summary["pass"].all() else "LIMITED / FAIL"

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for sampling, group in monitoring_bias.groupby("sampling"):
    axes[0].plot(group["monitoring_steps"], group["bias_vs_analytic"], marker="o", label=sampling)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set(xscale="log", xlabel="Monitoring steps M", ylabel="Direct price - analytic", title="Monitoring bias")
axes[0].legend()
for method, group in greek_summary[greek_summary["greek"] == "Gamma"].groupby("method"):
    axes[1].plot(100 * group["bump"], group["mean_estimate"], marker="o", label=method)
analytic_gamma = greek_summary[(greek_summary["method"] == "Conditional weight") & (greek_summary["greek"] == "Gamma")]
axes[1].plot(100 * analytic_gamma["bump"], analytic_gamma["analytic_fd"], color="black", linestyle="--", label="Analytic FD")
axes[1].set(xlabel="Spot bump (%)", ylabel="Gamma", title="Gamma bump diagnostic")
axes[1].legend(fontsize=8)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "day3b_diagnostics.png", dpi=180)
plt.show()

input_table.to_csv(OUTPUT_DIR / "market_inputs.csv", index=False)
price_replications.to_csv(OUTPUT_DIR / "price_replications.csv", index=False)
price_summary.to_csv(OUTPUT_DIR / "price_summary.csv", index=False)
monitoring_bias.to_csv(OUTPUT_DIR / "monitoring_bias_summary.csv", index=False)
bridge_agreement.to_csv(OUTPUT_DIR / "bridge_method_agreement.csv", index=False)
fine_grid_reference.to_csv(OUTPUT_DIR / "fine_grid_nested_mc.csv", index=False)
sampling_convergence.to_csv(OUTPUT_DIR / "sampling_convergence.csv", index=False)
greek_replications.to_csv(OUTPUT_DIR / "greek_replications.csv", index=False)
greek_summary.to_csv(OUTPUT_DIR / "greek_summary.csv", index=False)
greek_stability.to_csv(OUTPUT_DIR / "greek_stability.csv", index=False)
conditioning_rqmc_gain.to_csv(OUTPUT_DIR / "conditioning_rqmc_gain.csv", index=False)
gate_summary.to_csv(OUTPUT_DIR / "gate_summary.csv", index=False)

run_manifest = pd.DataFrame([{
    "run_date": pd.Timestamp.now().isoformat(),
    "gate_status": gate_status,
    "project_root": str(PROJECT_DIR),
    "config": str(CONFIG_FILE.relative_to(PROJECT_DIR)),
    "config_schema_version": PROJECT_CONFIG["schema_version"],
    "workbook": str(SOURCE_FILE.relative_to(PROJECT_DIR)),
    "workbook_sha256": SOURCE_SHA256,
    "underlying": market["ticker"],
    "model": "risk-neutral GBM",
    "product": "continuous lower-barrier down-and-out call",
    "S0": S0,
    "strike": STRIKE,
    "absolute_barrier": BARRIER,
    "maturity": T,
    "risk_free_rate": RISK_FREE_RATE,
    "dividend_yield": DIVIDEND_YIELD,
    "volatility": BASE_VOLATILITY,
    "analytic_price": ANALYTIC_PRICE,
    "monitoring_steps": ", ".join(map(str, MONITORING_STEPS)),
    "sample_sizes": ", ".join(map(str, SAMPLE_SIZES)),
    "price_replications": PRICE_REPLICATIONS,
    "fine_grid_steps": FINE_STEPS,
    "fine_grid_sample_size": FINE_N,
    "greek_monitoring_steps": GREEK_STEPS,
    "greek_sample_size": GREEK_N,
    "greek_replications": GREEK_REPLICATIONS,
    "price_runtime_seconds": price_runtime,
    "greek_runtime_seconds": greek_runtime,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
}])
run_manifest.to_csv(OUTPUT_DIR / "run_manifest.csv", index=False)
display(gate_summary)
display(conditioning_rqmc_gain)
print(f"Day 3B gate: {gate_status}")
print(f"Saved evidence to {OUTPUT_DIR}")

## Takeaways

- Brownian-bridge conditioning changes the monitoring convention: both exact
  bridge estimators target continuous monitoring even on a coarse endpoint grid.
- Direct endpoint monitoring misses intra-step barrier hits, so it is expected
  to overprice this down-and-out claim; increasing \(M\) reduces that bias.
- Increasing \(N\) reduces sampling uncertainty but does not remove monitoring
  bias. These are separate convergence axes.
- The weighted estimator integrates out the bridge indicator and is therefore
  usually the smoother input for RQMC and finite-difference Greeks.